# CPU Cluster: Distributed Hyperparameter Optimization with Ray Tune

This notebook trains **90 ML models** using **Ray Tune** for distributed hyperparameter optimization on a CPU cluster:
- Logistic Regression (15 models)
- LinearSVC (10 models) - fast linear SVM with O(n) complexity
- Random Forest (20 models)
- XGBoost (20 models)
- LightGBM Gradient Boosting (15 models)
- Naive Bayes (10 models)

**Key Feature**: Uses Ray Tune for distributed hyperparameter optimization with MLflow integration.

**Cluster Configuration**: 8 workers, 32 cores per node (256 total cores)

**Note**: Run this notebook on the CPU cluster.

## Prerequisites

- **Cluster access mode**: Dedicated (formerly single user) or No isolation shared access modes

## 1. Setup and Imports

In [ ]:
# Core libraries
import numpy as np
import pandas as pd
from datetime import datetime
import json
import time
import os

# Ray imports
import ray
from ray import tune
from ray.tune.search.optuna import OptunaSearch
from ray.tune.schedulers import ASHAScheduler
from ray.air.integrations.mlflow import MLflowLoggerCallback

# Scikit-learn models and utilities
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, precision_score, recall_score
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif, mutual_info_classif

# XGBoost
import xgboost as xgb

# LightGBM
import lightgbm as lgb

# PySpark
from pyspark.sql import functions as F

# MLflow for model logging and registry
import mlflow
import mlflow.sklearn
import mlflow.xgboost
import mlflow.lightgbm
from mlflow.models.signature import infer_signature
from mlflow.utils.databricks_utils import get_databricks_env_vars

print("All imports successful!")
print(f"Notebook type: CPU CLUSTER with Ray Tune Distributed HPO")

## 2. Configuration

In [ ]:
catalog = "ryuta"
schema = "ray"
experiment_name = "/Users/ryuta.yoshimatsu@databricks.com/ray_cpu_model_training_ray_tune"


print(f"Running with parameters:")
print(f"  Catalog: {catalog}")
print(f"  Schema: {schema}")
print(f"  Experiment: {experiment_name}")

# Configuration
CONFIG = {
    'catalog': catalog,
    'schema': schema,
    'table_name': f'{catalog}.{schema}.synthetic_data',
    'results_table': f'{catalog}.{schema}.model_training_results',
    'test_size': 0.2,
    'random_state': 42,
    'n_trials_per_model': 10,  # Ray Tune trials per model
    
    # Cluster configuration
    'cluster_type': 'cpu',
    'n_workers': 8,
    'cores_per_head_node': 0,
    'cores_per_node': 32,
    'total_cores': 256,
    
    # Model distribution (CPU models only)
    'model_distribution': {
        'logistic_regression': 15,
        'svm': 10,
        'random_forest': 20,
        'xgboost': 20,
        'lgbm': 15,
        'naive_bayes': 10
    },
    
    # Model ID offset (CPU models: 0-89)
    'model_id_start': 0,
    
    # MLflow Unity Catalog model registry
    'model_registry_path': f'{catalog}.{schema}',  # catalog.schema
    'experiment_name': experiment_name
}

CONFIG['n_models_total'] = sum(CONFIG['model_distribution'].values())

# Set up MLflow to use Unity Catalog
mlflow.set_registry_uri("databricks-uc")
mlflow.set_experiment(CONFIG['experiment_name'])

# Get Databricks credentials for Ray workers
# These will be passed to each Ray task to enable MLflow logging
mlflow_db_creds = get_databricks_env_vars("databricks")

print("Configuration loaded successfully!")
print(json.dumps({k: v for k, v in CONFIG.items() if k != 'model_distribution'}, indent=2))
print(f"Model distribution: {CONFIG['model_distribution']}")
print(f"\nMLflow registry URI: {mlflow.get_registry_uri()}")
print(f"MLflow experiment: {CONFIG['experiment_name']}")
print(f"\nDatabricks credentials captured for Ray workers: {list(mlflow_db_creds.keys())}")

## 3. Load Data from Delta Table

In [ ]:
# Load data from Delta table
print(f"Loading data from {CONFIG['table_name']}...")
df_spark = spark.table(CONFIG['table_name'])

print(f"Total rows: {df_spark.count()}")

# Convert to pandas
df = df_spark.toPandas()
print(f"Data loaded successfully! Shape: {df.shape}")

# Prepare features and labels
feature_columns = [col for col in df.columns if col.startswith('feature_')]
X = df[feature_columns].values
y = df['label'].values

print(f"Features shape: {X.shape}")
print(f"Labels shape: {y.shape}")
print(f"Class distribution: {np.bincount(y)}")

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=CONFIG['test_size'], 
    random_state=CONFIG['random_state'],
    stratify=y
)

print(f"\nTrain set: {X_train.shape}")
print(f"Test set: {X_test.shape}")

## 4. Feature Subset Selection

In [ ]:
from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType

def generate_feature_subsets_distributed(n_features, n_subsets, X_train_array, y_train_array, spark_session):
    """
    Generate diverse feature subsets using distributed computation via Pandas UDF.
    Feature importance scores (F-score and MI) are computed in parallel across the cluster.
    """
    np.random.seed(CONFIG['random_state'])
    random_state = CONFIG['random_state']
    
    print("Computing feature importance scores using distributed Pandas UDF...")
    start_time = time.time()
    
    # Broadcast training data to all workers
    X_train_bc = spark_session.sparkContext.broadcast(X_train_array)
    y_train_bc = spark_session.sparkContext.broadcast(y_train_array)
    
    # Create a DataFrame with feature indices
    feature_indices_df = spark_session.createDataFrame(
        [(i,) for i in range(n_features)],
        schema=StructType([StructField("feature_idx", IntegerType(), False)])
    )
    
    # Define the output schema for feature scores
    output_schema = StructType([
        StructField("feature_idx", IntegerType(), False),
        StructField("f_score", DoubleType(), False),
        StructField("mi_score", DoubleType(), False)
    ])
    
    def compute_feature_scores(iterator):
        """Compute F-score and MI score for each feature in the partition"""
        # Access broadcast variables
        X_train_local = X_train_bc.value
        y_train_local = y_train_bc.value
        
        for pdf in iterator:
            results = []
            for idx in pdf['feature_idx'].values:
                # Extract single feature column
                feature_values = X_train_local[:, idx].reshape(-1, 1)
                
                # Compute F-score (ANOVA F-value)
                f_score_val, _ = f_classif(feature_values, y_train_local)
                
                # Compute Mutual Information score
                mi_score_val = mutual_info_classif(
                    feature_values, y_train_local,
                    random_state=random_state
                )
                
                results.append({
                    'feature_idx': int(idx),
                    'f_score': float(f_score_val[0]) if not np.isnan(f_score_val[0]) else 0.0,
                    'mi_score': float(mi_score_val[0]) if not np.isnan(mi_score_val[0]) else 0.0
                })
            
            yield pd.DataFrame(results)
    
    # Repartition to distribute work across cluster and compute scores
    num_partitions = min(n_features, 100)  # Use reasonable number of partitions
    scores_df = (
        feature_indices_df
        .repartition(num_partitions)
        .mapInPandas(compute_feature_scores, schema=output_schema)
    )
    
    # Collect and sort scores
    scores_pandas = scores_df.orderBy("feature_idx").toPandas()
    f_scores = scores_pandas['f_score'].values
    mi_scores = scores_pandas['mi_score'].values
    
    # Clean up broadcast variables
    X_train_bc.unpersist()
    y_train_bc.unpersist()
    
    compute_time = time.time() - start_time
    print(f"Feature importance computation completed in {compute_time:.2f}s")
    
    # Sort features by scores (descending)
    top_f_features = np.argsort(f_scores)[::-1]
    top_mi_features = np.argsort(mi_scores)[::-1]
    
    # Generate subsets using various strategies
    subsets = []
    for i in range(n_subsets):
        subset_size = np.random.randint(20, n_features + 1)
        strategy = i % 7
        
        if strategy == 0:
            subset = list(range(n_features))
        elif strategy == 1:
            subset = sorted([int(x) for x in np.random.choice(n_features, subset_size, replace=False)])
        elif strategy == 2:
            subset = sorted([int(x) for x in top_f_features[:subset_size]])
        elif strategy == 3:
            subset = sorted([int(x) for x in top_mi_features[:subset_size]])
        elif strategy == 4:
            start = int(np.random.randint(0, n_features - subset_size + 1))
            subset = list(range(start, start + subset_size))
        elif strategy == 5:
            n_top = subset_size // 2
            top_features = [int(x) for x in top_f_features[:n_top]]
            remaining = [f for f in range(n_features) if f not in top_features]
            random_features = [int(x) for x in np.random.choice(remaining, subset_size - n_top, replace=False)]
            subset = sorted(top_features + random_features)
        else:
            n_top = subset_size // 2
            top_features = [int(x) for x in top_mi_features[:n_top]]
            remaining = [f for f in range(n_features) if f not in top_features]
            random_features = [int(x) for x in np.random.choice(remaining, subset_size - n_top, replace=False)]
            subset = sorted(top_features + random_features)
        
        subsets.append({
            'feature_indices': subset,
            'n_features': len(subset),
            'strategy': ['all', 'random', 'top_f', 'top_mi', 'block', 'f_random_mix', 'mi_random_mix'][strategy]
        })
    
    return subsets

# Generate feature subsets using distributed computation
n_features = X.shape[1]
feature_subsets = generate_feature_subsets_distributed(
    n_features, 
    CONFIG['n_models_total'], 
    X_train, 
    y_train, 
    spark
)

print(f"Generated {len(feature_subsets)} feature subsets")

## 5. Ray Tune Search Spaces

Define hyperparameter search spaces using Ray Tune's API instead of Optuna.

In [ ]:
# Ray Tune search spaces for CPU models
def get_ray_tune_search_space(model_type):
    """Get Ray Tune search space for a given model type"""
    
    if model_type == 'logistic_regression':
        return {
            'C': tune.loguniform(1e-4, 1e2),
            'penalty': tune.choice(['l2']),
            'solver': tune.choice(['lbfgs', 'saga']),
            'max_iter': tune.choice([500])
        }
    
    elif model_type == 'svm':
        return {
            'C': tune.loguniform(1e-3, 1e2),
            'penalty': tune.choice(['l2']),
            'loss': tune.choice(['squared_hinge']),
            'max_iter': tune.choice([2000]),
            'dual': tune.choice(['auto'])
        }
    
    elif model_type == 'random_forest':
        return {
            'n_estimators': tune.randint(50, 301),
            'max_depth': tune.randint(5, 31),
            'min_samples_split': tune.randint(2, 21),
            'min_samples_leaf': tune.randint(1, 11),
            'max_features': tune.choice(['sqrt', 'log2', None])
        }
    
    elif model_type == 'xgboost':
        return {
            'n_estimators': tune.randint(50, 301),
            'max_depth': tune.randint(3, 16),
            'learning_rate': tune.loguniform(1e-3, 0.3),
            'subsample': tune.uniform(0.5, 1.0),
            'colsample_bytree': tune.uniform(0.5, 1.0),
            'min_child_weight': tune.randint(1, 11),
            'gamma': tune.uniform(0, 5)
        }
    
    elif model_type == 'lgbm':
        return {
            'n_estimators': tune.randint(50, 301),
            'max_depth': tune.randint(3, 16),
            'learning_rate': tune.loguniform(1e-3, 0.3),
            'subsample': tune.uniform(0.5, 1.0),
            'colsample_bytree': tune.uniform(0.5, 1.0),
            'num_leaves': tune.randint(20, 101),
            'min_child_samples': tune.randint(5, 51),
            'reg_alpha': tune.loguniform(1e-8, 1.0),
            'reg_lambda': tune.loguniform(1e-8, 1.0)
        }
    
    elif model_type == 'naive_bayes':
        return {
            'var_smoothing': tune.loguniform(1e-11, 1e-7)
        }
    
    else:
        raise ValueError(f"Unknown model type: {model_type}")

print("Ray Tune search spaces defined!")

## 6. Training Functions

In [ ]:
def train_sklearn_model(model_type, hyperparams, X_tr, y_tr, X_val, y_val, random_state, return_model=False, n_cpus=1):
    """Train a scikit-learn model
    
    Args:
        n_cpus: Number of CPUs to use for parallelization (default=1, use explicit value to avoid resource contention)
    """
    
    if model_type == 'logistic_regression':
        model = LogisticRegression(**hyperparams, random_state=random_state)
    elif model_type == 'svm':
        # Use LinearSVC wrapped with CalibratedClassifierCV for probability estimates
        base_svm = LinearSVC(**hyperparams, random_state=random_state)
        model = CalibratedClassifierCV(base_svm, cv=3)
    elif model_type == 'random_forest':
        # Use explicit n_cpus instead of -1 to avoid resource contention in Ray tasks
        model = RandomForestClassifier(**hyperparams, random_state=random_state, n_jobs=n_cpus)
    elif model_type == 'naive_bayes':
        model = GaussianNB(**hyperparams)
    else:
        raise ValueError(f"Unknown model type: {model_type}")
    
    # Scale features
    scaler = StandardScaler()
    X_tr_scaled = scaler.fit_transform(X_tr)
    X_val_scaled = scaler.transform(X_val)
    
    # Train
    model.fit(X_tr_scaled, y_tr)
    
    # Predict
    y_pred = model.predict(X_val_scaled)
    y_pred_proba = model.predict_proba(X_val_scaled)[:, 1]
    
    # Calculate metrics
    metrics = {
        'accuracy': accuracy_score(y_val, y_pred),
        'roc_auc': roc_auc_score(y_val, y_pred_proba),
        'f1': f1_score(y_val, y_pred),
        'precision': precision_score(y_val, y_pred),
        'recall': recall_score(y_val, y_pred)
    }
    
    if return_model:
        return metrics, model, scaler
    return metrics


def train_xgboost_model(hyperparams, X_tr, y_tr, X_val, y_val, random_state, return_model=False, n_cpus=1):
    """Train XGBoost model
    
    Args:
        n_cpus: Number of CPUs to use for parallelization (default=1, use explicit value to avoid resource contention)
    """
    
    scaler = StandardScaler()
    X_tr_scaled = scaler.fit_transform(X_tr)
    X_val_scaled = scaler.transform(X_val)
    
    # Use explicit n_cpus instead of -1 to avoid resource contention in Ray tasks
    model = xgb.XGBClassifier(
        **hyperparams,
        objective='binary:logistic',
        eval_metric='auc',
        random_state=random_state,
        n_jobs=n_cpus,
        use_label_encoder=False
    )
    
    model.fit(X_tr_scaled, y_tr, eval_set=[(X_val_scaled, y_val)], verbose=False)
    
    y_pred = model.predict(X_val_scaled)
    y_pred_proba = model.predict_proba(X_val_scaled)[:, 1]
    
    metrics = {
        'accuracy': accuracy_score(y_val, y_pred),
        'roc_auc': roc_auc_score(y_val, y_pred_proba),
        'f1': f1_score(y_val, y_pred),
        'precision': precision_score(y_val, y_pred),
        'recall': recall_score(y_val, y_pred)
    }
    
    if return_model:
        return metrics, model, scaler
    return metrics


def train_lgbm_model(hyperparams, X_tr, y_tr, X_val, y_val, random_state, return_model=False, n_cpus=1):
    """Train LightGBM model
    
    Args:
        n_cpus: Number of CPUs to use for parallelization (default=1, use explicit value to avoid resource contention)
    """
    
    scaler = StandardScaler()
    X_tr_scaled = scaler.fit_transform(X_tr)
    X_val_scaled = scaler.transform(X_val)
    
    # Use explicit n_cpus instead of -1 to avoid resource contention in Ray tasks
    model = lgb.LGBMClassifier(
        **hyperparams,
        random_state=random_state,
        n_jobs=n_cpus,
        verbose=-1  # Suppress LightGBM output
    )
    
    model.fit(X_tr_scaled, y_tr, eval_set=[(X_val_scaled, y_val)])
    
    y_pred = model.predict(X_val_scaled)
    y_pred_proba = model.predict_proba(X_val_scaled)[:, 1]
    
    metrics = {
        'accuracy': accuracy_score(y_val, y_pred),
        'roc_auc': roc_auc_score(y_val, y_pred_proba),
        'f1': f1_score(y_val, y_pred),
        'precision': precision_score(y_val, y_pred),
        'recall': recall_score(y_val, y_pred)
    }
    
    if return_model:
        return metrics, model, scaler
    return metrics

print("Training functions defined!")

## 7. Ray Tune Trainable Function

Define the trainable function that Ray Tune will use for distributed hyperparameter optimization.

In [ ]:
def ray_tune_trainable(config, data_dict):
    """
    Ray Tune trainable function for hyperparameter optimization.
    
    Args:
        config: Dictionary containing hyperparameters sampled by Ray Tune
        data_dict: Dictionary containing training/validation data and metadata
    """
    from ray import train
    
    # Extract data from data_dict
    X_tr = data_dict['X_tr']
    y_tr = data_dict['y_tr']
    X_val = data_dict['X_val']
    y_val = data_dict['y_val']
    model_type = data_dict['model_type']
    random_state = data_dict['random_state']
    n_cpus = data_dict.get('n_cpus', 1)
    
    try:
        # Train model based on type
        if model_type == 'xgboost':
            metrics = train_xgboost_model(config, X_tr, y_tr, X_val, y_val, random_state, n_cpus=n_cpus)
        elif model_type == 'lgbm':
            metrics = train_lgbm_model(config, X_tr, y_tr, X_val, y_val, random_state, n_cpus=n_cpus)
        else:
            metrics = train_sklearn_model(model_type, config, X_tr, y_tr, X_val, y_val, random_state, n_cpus=n_cpus)
        
        # Report metrics to Ray Tune
        train.report({
            'roc_auc': metrics['roc_auc'],
            'accuracy': metrics['accuracy'],
            'f1': metrics['f1'],
            'precision': metrics['precision'],
            'recall': metrics['recall']
        })
        
    except Exception as e:
        # Report failure with low score
        train.report({
            'roc_auc': 0.5,
            'accuracy': 0.5,
            'f1': 0.0,
            'precision': 0.0,
            'recall': 0.0,
            'error': str(e)
        })

print("Ray Tune trainable function defined!")

## 8. Ray Remote Training Function with Ray Tune HPO and MLflow

This function combines Ray Tune for hyperparameter optimization with MLflow logging.

In [ ]:
def convert_numpy_types(obj):
    """Convert numpy types to native Python for JSON serialization"""
    if isinstance(obj, dict):
        return {k: convert_numpy_types(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [convert_numpy_types(v) for v in obj]
    elif isinstance(obj, (np.integer,)):
        return int(obj)
    elif isinstance(obj, (np.floating,)):
        return float(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    return obj


@ray.remote
def train_model_with_ray_tune(
    model_id,
    model_type,
    feature_subset,
    X_train_full,
    y_train_full,
    X_test_full,
    y_test_full,
    parent_run_id,
    mlflow_db_creds,
    experiment_name,
    model_registry_path,
    random_state,
    n_trials=10,
    n_cpus=1
):
    """
    Ray remote function to train a single model with Ray Tune hyperparameter optimization.
    
    Uses Ray Tune for distributed HPO and logs results to MLflow as a child run.
    """
    import os
    import time
    import json
    import numpy as np
    import mlflow
    import mlflow.sklearn
    import mlflow.xgboost
    import mlflow.lightgbm
    from mlflow.models.signature import infer_signature
    from sklearn.model_selection import train_test_split
    from sklearn.preprocessing import StandardScaler
    from sklearn.linear_model import LogisticRegression
    from sklearn.svm import LinearSVC
    from sklearn.calibration import CalibratedClassifierCV
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.naive_bayes import GaussianNB
    from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, precision_score, recall_score
    import xgboost as xgb
    import lightgbm as lgb
    from ray import tune, train as ray_train
    from ray.tune.search.optuna import OptunaSearch
    
    start_time = time.time()
    
    try:
        # Set MLflow credentials within the Ray task
        os.environ.update(mlflow_db_creds)
        
        # Set MLflow to use Unity Catalog and the correct experiment
        mlflow.set_registry_uri("databricks-uc")
        mlflow.set_experiment(experiment_name)
        
        # Extract feature subset
        feature_indices = feature_subset['feature_indices']
        X_train = X_train_full[:, feature_indices]
        X_test = X_test_full[:, feature_indices]
        
        # Split for validation
        X_tr, X_val, y_tr, y_val = train_test_split(
            X_train, y_train_full,
            test_size=0.2,
            random_state=random_state,
            stratify=y_train_full
        )
        
        # Define training function for Ray Tune
        def tune_trainable(config):
            from ray import train as ray_train
            
            try:
                # Train model based on type
                scaler = StandardScaler()
                X_tr_scaled = scaler.fit_transform(X_tr)
                X_val_scaled = scaler.transform(X_val)
                
                if model_type == 'logistic_regression':
                    model = LogisticRegression(**config, random_state=random_state)
                elif model_type == 'svm':
                    base_svm = LinearSVC(**config, random_state=random_state)
                    model = CalibratedClassifierCV(base_svm, cv=3)
                elif model_type == 'random_forest':
                    model = RandomForestClassifier(**config, random_state=random_state, n_jobs=1)
                elif model_type == 'naive_bayes':
                    model = GaussianNB(**config)
                elif model_type == 'xgboost':
                    model = xgb.XGBClassifier(
                        **config,
                        objective='binary:logistic',
                        eval_metric='auc',
                        random_state=random_state,
                        n_jobs=1,
                        use_label_encoder=False
                    )
                elif model_type == 'lgbm':
                    model = lgb.LGBMClassifier(
                        **config,
                        random_state=random_state,
                        n_jobs=1,
                        verbose=-1
                    )
                else:
                    raise ValueError(f"Unknown model type: {model_type}")
                
                # Train
                if model_type == 'xgboost':
                    model.fit(X_tr_scaled, y_tr, eval_set=[(X_val_scaled, y_val)], verbose=False)
                elif model_type == 'lgbm':
                    # LightGBM doesn't accept verbose in fit(), it's set in constructor
                    model.fit(X_tr_scaled, y_tr, eval_set=[(X_val_scaled, y_val)])
                else:
                    model.fit(X_tr_scaled, y_tr)
                
                # Predict
                y_pred = model.predict(X_val_scaled)
                y_pred_proba = model.predict_proba(X_val_scaled)[:, 1]
                
                # Calculate metrics
                roc_auc = roc_auc_score(y_val, y_pred_proba)
                accuracy = accuracy_score(y_val, y_pred)
                
                ray_train.report({'roc_auc': roc_auc, 'accuracy': accuracy})
                
            except Exception as e:
                ray_train.report({'roc_auc': 0.5, 'accuracy': 0.5, 'error': str(e)})
        
        # Get search space for this model type
        if model_type == 'logistic_regression':
            search_space = {
                'C': tune.loguniform(1e-4, 1e2),
                'penalty': tune.choice(['l2']),
                'solver': tune.choice(['lbfgs', 'saga']),
                'max_iter': tune.choice([500])
            }
        elif model_type == 'svm':
            search_space = {
                'C': tune.loguniform(1e-3, 1e2),
                'penalty': tune.choice(['l2']),
                'loss': tune.choice(['squared_hinge']),
                'max_iter': tune.choice([2000]),
                'dual': tune.choice(['auto'])
            }
        elif model_type == 'random_forest':
            search_space = {
                'n_estimators': tune.randint(50, 301),
                'max_depth': tune.randint(5, 31),
                'min_samples_split': tune.randint(2, 21),
                'min_samples_leaf': tune.randint(1, 11),
                'max_features': tune.choice(['sqrt', 'log2', None])
            }
        elif model_type == 'xgboost':
            search_space = {
                'n_estimators': tune.randint(50, 301),
                'max_depth': tune.randint(3, 16),
                'learning_rate': tune.loguniform(1e-3, 0.3),
                'subsample': tune.uniform(0.5, 1.0),
                'colsample_bytree': tune.uniform(0.5, 1.0),
                'min_child_weight': tune.randint(1, 11),
                'gamma': tune.uniform(0, 5)
            }
        elif model_type == 'lgbm':
            search_space = {
                'n_estimators': tune.randint(50, 301),
                'max_depth': tune.randint(3, 16),
                'learning_rate': tune.loguniform(1e-3, 0.3),
                'subsample': tune.uniform(0.5, 1.0),
                'colsample_bytree': tune.uniform(0.5, 1.0),
                'num_leaves': tune.randint(20, 101),
                'min_child_samples': tune.randint(5, 51),
                'reg_alpha': tune.loguniform(1e-8, 1.0),
                'reg_lambda': tune.loguniform(1e-8, 1.0)
            }
        elif model_type == 'naive_bayes':
            search_space = {
                'var_smoothing': tune.loguniform(1e-11, 1e-7)
            }
        else:
            raise ValueError(f"Unknown model type: {model_type}")
        
        # Run Ray Tune optimization
        tuner = tune.Tuner(
            tune_trainable,
            param_space=search_space,
            tune_config=tune.TuneConfig(
                metric='roc_auc',
                mode='max',
                num_samples=n_trials,
                search_alg=OptunaSearch(seed=random_state),
            ),
            run_config=ray_train.RunConfig(
                verbose=0,
                # Disable checkpointing to avoid file race conditions on distributed filesystem
                checkpoint_config=ray_train.CheckpointConfig(
                    checkpoint_frequency=0,
                    checkpoint_at_end=False,
                ),
            )
        )
        
        results = tuner.fit()
        
        # Get best result
        best_result = results.get_best_result(metric='roc_auc', mode='max')
        best_hyperparams = best_result.config
        best_val_score = best_result.metrics['roc_auc']
        
        # Train final model on full training set with best hyperparams
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)
        
        if model_type == 'logistic_regression':
            final_model = LogisticRegression(**best_hyperparams, random_state=random_state)
        elif model_type == 'svm':
            base_svm = LinearSVC(**best_hyperparams, random_state=random_state)
            final_model = CalibratedClassifierCV(base_svm, cv=3)
        elif model_type == 'random_forest':
            final_model = RandomForestClassifier(**best_hyperparams, random_state=random_state, n_jobs=n_cpus)
        elif model_type == 'naive_bayes':
            final_model = GaussianNB(**best_hyperparams)
        elif model_type == 'xgboost':
            final_model = xgb.XGBClassifier(
                **best_hyperparams,
                objective='binary:logistic',
                eval_metric='auc',
                random_state=random_state,
                n_jobs=n_cpus,
                use_label_encoder=False
            )
        elif model_type == 'lgbm':
            final_model = lgb.LGBMClassifier(
                **best_hyperparams,
                random_state=random_state,
                n_jobs=n_cpus,
                verbose=-1
            )
        
        # Train final model
        if model_type == 'xgboost':
            final_model.fit(X_train_scaled, y_train_full, eval_set=[(X_test_scaled, y_test_full)], verbose=False)
        elif model_type == 'lgbm':
            # LightGBM doesn't accept verbose in fit(), it's set in constructor
            final_model.fit(X_train_scaled, y_train_full, eval_set=[(X_test_scaled, y_test_full)])
        else:
            final_model.fit(X_train_scaled, y_train_full)
        
        # Get test metrics
        y_pred = final_model.predict(X_test_scaled)
        y_pred_proba = final_model.predict_proba(X_test_scaled)[:, 1]
        
        test_metrics = {
            'accuracy': accuracy_score(y_test_full, y_pred),
            'roc_auc': roc_auc_score(y_test_full, y_pred_proba),
            'f1': f1_score(y_test_full, y_pred),
            'precision': precision_score(y_test_full, y_pred),
            'recall': recall_score(y_test_full, y_pred)
        }
        
        training_time = time.time() - start_time
        
        # Log to MLflow as a child run
        registered_model_name = f"{model_registry_path}.cpu_model_{model_id}_ray_tune"
        
        # Create a nested child run associated with the parent run_id
        with mlflow.start_run(run_name=f"cpu_model_{model_id}_{model_type}_ray_tune", parent_run_id=parent_run_id, nested=True) as child_run:
            # Log parameters
            mlflow.log_params(convert_numpy_types(best_hyperparams))
            mlflow.log_param("model_type", model_type)
            mlflow.log_param("model_id", model_id)
            mlflow.log_param("n_features_used", len(feature_indices))
            mlflow.log_param("feature_strategy", feature_subset['strategy'])
            mlflow.log_param("n_trials", n_trials)
            mlflow.log_param("hpo_method", "ray_tune")
            
            # Log metrics
            mlflow.log_metrics({
                'accuracy': float(test_metrics['accuracy']),
                'roc_auc': float(test_metrics['roc_auc']),
                'f1': float(test_metrics['f1']),
                'precision': float(test_metrics['precision']),
                'recall': float(test_metrics['recall']),
                'best_val_score': float(best_val_score),
                'training_time': float(training_time)
            })
            
            # Log feature indices as artifact
            feature_indices_str = json.dumps(convert_numpy_types(feature_indices))
            mlflow.log_text(feature_indices_str, "feature_indices.json")
            
            # Create model signature
            signature = infer_signature(X_test_scaled, final_model.predict(X_test_scaled))
            
            # Log and register model based on type
            if model_type == 'xgboost':
                mlflow.xgboost.log_model(
                    final_model,
                    artifact_path="model",
                    signature=signature,
                    registered_model_name=registered_model_name
                )
            elif model_type == 'lgbm':
                mlflow.lightgbm.log_model(
                    final_model,
                    artifact_path="model",
                    signature=signature,
                    registered_model_name=registered_model_name
                )
            else:
                mlflow.sklearn.log_model(
                    final_model,
                    artifact_path="model",
                    signature=signature,
                    registered_model_name=registered_model_name
                )
            
            child_run_id = child_run.info.run_id
        
        # Return result summary
        result = {
            'model_id': model_id,
            'model_type': model_type,
            'cluster_type': 'cpu',
            'hpo_method': 'ray_tune',
            'n_features_used': len(feature_indices),
            'feature_strategy': feature_subset['strategy'],
            'best_hyperparams': json.dumps(convert_numpy_types(best_hyperparams)),
            'best_val_score': float(best_val_score),
            'accuracy': float(test_metrics['accuracy']),
            'roc_auc': float(test_metrics['roc_auc']),
            'f1': float(test_metrics['f1']),
            'precision': float(test_metrics['precision']),
            'recall': float(test_metrics['recall']),
            'training_time': float(training_time),
            'mlflow_run_id': child_run_id,
            'registered_model_name': registered_model_name,
            'status': 'success'
        }
        
        return result
        
    except Exception as e:
        import traceback
        return {
            'model_id': model_id,
            'model_type': model_type,
            'cluster_type': 'cpu',
            'hpo_method': 'ray_tune',
            'status': 'failed',
            'error': str(e),
            'traceback': traceback.format_exc(),
            'training_time': time.time() - start_time
        }

print("Ray remote training function with Ray Tune HPO defined!")

## 9. Initialize Ray

In [ ]:
# Initialize Ray cluster using Databricks utilities
from ray.util.spark import setup_ray_cluster, shutdown_ray_cluster

# Shutdown any existing Ray instance
if ray.is_initialized():
    ray.shutdown()

try:
    # Setup Ray cluster spanning all Spark nodes
    # autoscale=False ensures fixed cluster size (no dynamic scaling)
    print("Setting up Ray cluster across all nodes (autoscaling disabled)...")
    setup_ray_cluster(
        min_worker_nodes=CONFIG['n_workers'],  # 8 worker nodes
        max_worker_nodes=CONFIG['n_workers'],  # 8 worker nodes
        num_cpus_head_node=CONFIG['cores_per_head_node'], # 16 CPUs per head node
        num_cpus_worker_node=CONFIG['cores_per_node'],  # 32 CPUs per worker node
        num_gpus_head_node=0,   # 0 GPUs for CPU cluster
        num_gpus_worker_node=0,  # 0 GPUs for CPU cluster
        collect_log_to_path="/Workspace/Users/ryuta.yoshimatsu@databricks.com/ray_logs",
    )
    # Explicitly initialize Ray after cluster setup
    ray.init(
        address='auto', 
        ignore_reinit_error=True, 
        logging_level='ERROR',
    )
    
except Exception as e:
    print(f"Warning: setup_ray_cluster failed with: {e}")
    print("Falling back to standard Ray initialization...")    
    ray.init(
        ignore_reinit_error=True, 
        logging_level='ERROR',
    )

print("\nRay initialized successfully on CPU cluster!")
print(f"Available CPUs: {ray.cluster_resources().get('CPU', 0)}")
print(f"Available GPUs: {ray.cluster_resources().get('GPU', 0)}")
print(f"Expected CPUs: {CONFIG['total_cores']} (8 workers × 32 cores)")

# Verify cluster setup
print(f"\nCluster nodes connected: {len(ray.nodes())}")
print(f"Total cluster resources: {ray.cluster_resources()}")

## 10. Generate Model Configurations

In [ ]:
# Generate model configurations
model_configs = []
model_id = CONFIG['model_id_start']

for model_type, count in CONFIG['model_distribution'].items():
    for i in range(count):
        feature_subset = feature_subsets[model_id % len(feature_subsets)]
        
        # Determine CPU allocation
        # Models with parallelization support (n_jobs) get 4 CPUs
        if model_type in ['random_forest', 'xgboost', 'lgbm']:
            n_cpus = 4
        else:
            n_cpus = 1
        
        config = {
            'model_id': model_id,
            'model_type': model_type,
            'feature_subset': feature_subset,
            'n_cpus': n_cpus
        }
        
        model_configs.append(config)
        model_id += 1

print(f"Generated {len(model_configs)} CPU model configurations")
print(f"Model IDs: {CONFIG['model_id_start']} to {model_id - 1}")
print(f"\nModel distribution:")
for model_type, count in CONFIG['model_distribution'].items():
    print(f"  {model_type}: {count}")

## 11. Launch Parallel Training with Ray Tune HPO and MLflow Parent Run

We create a **parent run** on the driver, then pass its `run_id` to all Ray tasks.
Each Ray task uses **Ray Tune** for hyperparameter optimization and creates a **child run** under this parent.

In [ ]:
# Put data in Ray object store
X_train_ref = ray.put(X_train)
y_train_ref = ray.put(y_train)
X_test_ref = ray.put(X_test)
y_test_ref = ray.put(y_test)

print("Data stored in Ray object store")

In [ ]:
# Start parent run on the main driver process
# All Ray tasks will create child runs under this parent

print(f"\nStarting MLflow parent run and launching {len(model_configs)} training jobs with Ray Tune HPO...\n")
print("="*80)
start_time = time.time()

# Create the parent MLflow run
with mlflow.start_run(run_name="cpu_parallel_training_ray_tune_parent") as parent_run:
    parent_run_id = parent_run.info.run_id
    print(f"Parent MLflow run ID: {parent_run_id}")
    
    # Log parent run metadata
    mlflow.log_params({
        'n_models': len(model_configs),
        'cluster_type': CONFIG['cluster_type'],
        'n_workers': CONFIG['n_workers'],
        'total_cores': CONFIG['total_cores'],
        'n_trials_per_model': CONFIG['n_trials_per_model'],
        'hpo_method': 'ray_tune'
    })
    
    # Launch all training jobs as Ray tasks
    # Each task will use Ray Tune for HPO and create a child run under the parent
    futures = []
    for config in model_configs:
        remote_fn = train_model_with_ray_tune.options(num_cpus=config['n_cpus'])
        
        future = remote_fn.remote(
            model_id=config['model_id'],
            model_type=config['model_type'],
            feature_subset=config['feature_subset'],
            X_train_full=X_train_ref,
            y_train_full=y_train_ref,
            X_test_full=X_test_ref,
            y_test_full=y_test_ref,
            parent_run_id=parent_run_id,
            mlflow_db_creds=mlflow_db_creds,
            experiment_name=CONFIG['experiment_name'],
            model_registry_path=CONFIG['model_registry_path'],
            random_state=CONFIG['random_state'],
            n_trials=CONFIG['n_trials_per_model'],
            n_cpus=config['n_cpus']
        )
        
        futures.append(future)
    
    print(f"All {len(futures)} jobs submitted. Waiting for results...\n")
    
    # Collect results from Ray workers
    results = []
    completed = 0
    remaining_futures = futures.copy()
    
    print("Collecting training results from workers (each using Ray Tune HPO and logging to MLflow as child run)...")
    
    while remaining_futures:
        ready_futures, remaining_futures = ray.wait(remaining_futures, num_returns=1)
        
        for future in ready_futures:
            result = ray.get(future)
            results.append(result)
            completed += 1
            
            if result['status'] == 'success':
                print(f"[{completed}/{len(futures)}] Model {result['model_id']} ({result['model_type']}) - "
                      f"ROC AUC: {result['roc_auc']:.4f} ({result['training_time']:.1f}s) -> Ray Tune HPO + MLflow child run")
            else:
                print(f"[{completed}/{len(futures)}] Model {result['model_id']} FAILED: {result.get('error', 'Unknown')}")
    
    training_time = time.time() - start_time
    
    # Log summary metrics to parent run
    successful_results = [r for r in results if r['status'] == 'success']
    failed_results = [r for r in results if r['status'] != 'success']
    
    if successful_results:
        roc_aucs = [r['roc_auc'] for r in successful_results]
        accuracies = [r['accuracy'] for r in successful_results]
        
        mlflow.log_metrics({
            'total_training_time': training_time,
            'n_successful': len(successful_results),
            'n_failed': len(failed_results),
            'mean_roc_auc': np.mean(roc_aucs),
            'best_roc_auc': np.max(roc_aucs),
            'mean_accuracy': np.mean(accuracies),
            'best_accuracy': np.max(accuracies)
        })

print(f"\n{'='*80}")
print(f"ALL TRAINING JOBS COMPLETE")
print(f"{'='*80}")
print(f"Total time: {training_time:.2f}s ({training_time/60:.2f} minutes)")
print(f"Successful: {len(successful_results)}")
print(f"Failed: {len(failed_results)}")
print(f"\nParent MLflow run ID: {parent_run_id}")
print(f"All child runs logged under: {CONFIG['experiment_name']}")

In [ ]:
# Shutdown Ray cluster using shutdown_ray_cluster() for clean Spark integration
from ray.util.spark import shutdown_ray_cluster
import ray

print("\nShutting down Ray cluster...")
try:
    shutdown_ray_cluster()
    print("Ray cluster shut down successfully!")
except Exception as e:
    print(f"Warning: Ray cluster shutdown encountered an error (non-critical): {e}")
    print("Ray cluster will be automatically cleaned up when notebook is detached.")

# Also shutdown Ray client
try:
    ray.shutdown()
    print("Ray client shut down successfully!")
except Exception as e:
    print(f"Warning: Ray client shutdown encountered an error: {e}")

## Summary

This notebook trained **90 traditional ML models** using **Ray Tune for distributed hyperparameter optimization**:

**Key Architecture Changes (compared to ray_cpu_model_training.ipynb):**
- **Ray Tune** replaces Optuna for hyperparameter optimization
- Each Ray task runs Ray Tune internally for distributed HPO
- Uses OptunaSearch as the search algorithm within Ray Tune
- Parent MLflow run created on the driver process
- Each Ray task creates a child run under the parent
- MLflow credentials passed to Ray workers via `get_databricks_env_vars`

**Benefits of Ray Tune:**
- Native integration with Ray's distributed infrastructure
- Efficient resource allocation for HPO trials
- Support for advanced schedulers (ASHA, PBT, etc.)
- Better scalability for large-scale hyperparameter search

**Models registered to Unity Catalog:** `{catalog}.{schema}.cpu_model_*_ray_tune`

**Next Steps:**
1. View the parent run in MLflow to see all child runs
2. Compare model performance across child runs
3. Compare results with the Optuna-based approach